# Module 3 — Build vs Buy vs Fine-Tune, as a constraint cascade

**The problem.** "Which model?" is usually answered by a leaderboard screenshot and a vibe. The architect answers it as a **cascade of constraints**: a privacy gate (where may the data travel, and under what contract), then the capability the task actually needs, then economics against the *cheapest API tier that passes your eval*, and finally latency to place the deployment. Plus a **measured** prompted baseline before anyone says 'fine-tune'.

This is a cascade, **not** a weighted scorecard. Privacy and capability are hard gates, not scores you trade off against cost. Some considerations (operational complexity, exact capability-fit) need a human and an eval, not an attribute — this notebook models the parts that *are* decidable from attributes and is honest about the rest.

General tutorial. The companion `model-selection-memo` **skill** produces the decision memo for a real task. Needs Ollama (`qwen2.5:7b`) for the baseline-measurement section; the rest is pure Python.


## 0. Five considerations — two are gates, not scores

- **Data-privacy posture** — a *gate*: where may the data go, and under what contract (public API with a DPA, in-tenancy managed cloud, on-prem, on-device). Not tradeable against cost.
- **Task-capability fit** — a *gate*: does a model you can actually run (a self-hostable open-weight, a cheap API tier) pass *your* eval? Tested on your eval set, never a leaderboard rank.
- **Cost at production scale** — economics among the options that pass both gates (section 3).
- **Latency profile** — which latency (time-to-first-token vs total), at P95; can force an on-device locus.
- **Operational complexity** — API < managed cloud < self-host < fine-tune, rising ownership. Encoded in the locus ordering, but the real number is a team judgement, not an attribute.


## 1. The cascade, derived from task attributes (so it generalises)

The decision reads the task's *attributes*, never its name — add a task you've never seen and it still resolves. Order: approach (RAG vs prompt vs fine-tune-candidate) → latency gate (realtime+small → on-device) → privacy gate (contract + allowed loci) → capability (is self-host even an option?) → economics (compare against the cheapest passing tier).


In [1]:
from dataclasses import dataclass

@dataclass
class Task:
    name: str
    privacy: str          # public | internal | regulated   (PII/PHI/contractual)
    latency: str          # batch | interactive | realtime
    volume_month: int     # requests / month at launch
    capability_need: str  # low | medium | frontier
    has_labels: bool      # labelled examples available for fine-tuning/eval
    strict_format: bool   # needs a strict output schema
    knowledge_gap: bool   # needs facts outside the model's training

@dataclass
class Decision:
    approach_kind: str    # rag | finetune-candidate | prompt   (assert on THIS, not the prose)
    approach: str
    locus: str            # on-device | self-host | managed-cloud | api
    tier: str             # cheap | mid | frontier | open-weight | on-device
    contract: str         # none | DPA | BAA/DPA + zero-retention (or in-tenancy)
    why: str

# --- COST ASSUMPTIONS (illustrative, dated 2026-10; NOT live quotes) ---
API_TIERS = {"cheap": 0.0006, "mid": 0.004, "frontier": 0.015}   # USD / 1k tokens, blended in+out
TIER_FOR  = {"low": "cheap", "medium": "mid", "frontier": "frontier"}
TOKENS_PER_REQUEST      = 700
SELFHOST_USD_PER_MONTH  = 4_000       # one GPU box (amortised) + ops, always-on, open-weight model
SELFHOST_CAPACITY_MONTH = 8_000_000   # requests/month one box serves at this task size

def api_cost(reqs, tier):  return reqs * TOKENS_PER_REQUEST/1000 * API_TIERS[tier]
def selfhost_cost(reqs):   return max(1, -(-reqs // SELFHOST_CAPACITY_MONTH)) * SELFHOST_USD_PER_MONTH
def crossover(tier):       return SELFHOST_USD_PER_MONTH / (TOKENS_PER_REQUEST/1000 * API_TIERS[tier])

def approach(t):
    if t.knowledge_gap:
        return ("rag", "RAG (the gap is missing facts, not wrong behaviour -> retrieve, don't fine-tune)")
    if t.strict_format and t.has_labels:
        return ("finetune-candidate", "prompt baseline first; fine-tune is a CANDIDATE only if the "
                "measured baseline ceilings out")
    return ("prompt", "prompted LLM (measure the baseline before anything heavier)")

def decide(t):
    ak, ap = approach(t)
    # LATENCY GATE: realtime + small capability -> a specialised model on the device, no network hop
    if t.latency == "realtime" and t.capability_need == "low":
        return Decision(ak, ap,"on-device", "on-device", "none (data never leaves the device)",
                        "realtime + low capability -> small on-device/edge model; a network round-trip "
                        "to any API blows the latency budget")
    selfhostable = t.capability_need in ("low", "medium")   # a $4k open-weight box can't match frontier
    # PRIVACY GATE: where may data travel, and under what contract
    if t.privacy == "regulated":
        contract = "BAA/DPA + zero-retention (or keep it in-tenancy)"
        if not selfhostable:   # regulated + frontier -> frontier model INSIDE your tenancy
            return Decision(ak, ap,"managed-cloud", "frontier", contract,
                            "regulated + frontier -> a frontier model in your own cloud tenancy "
                            "(managed-cloud), OR a public API only under a signed BAA/DPA with zero "
                            "retention. 'No API regardless' is false — the question is the contract")
        return Decision(ak, ap,"self-host", "open-weight", contract,
                        "regulated + self-hostable capability -> run in-tenancy (cleanest), or a "
                        "contracted (BAA/DPA) API; pick on cost, not on a blanket ban")
    contract = "DPA" if t.privacy == "internal" else "none"
    if not selfhostable:       # frontier, not regulated -> no self-hostable open-weight passes the eval
        return Decision(ak, ap,"api", "frontier", contract,
                        "frontier capability with no self-hostable open-weight that passes -> managed "
                        "frontier / API; self-host isn't on the table until an open-weight passes your eval")
    # ECONOMICS: self-hostable capability, compare against the CHEAPEST tier that passes
    tier = TIER_FOR[t.capability_need]
    cx = crossover(tier)
    if cx < t.volume_month <= SELFHOST_CAPACITY_MONTH:
        return Decision(ak, ap,"self-host", "open-weight", contract,
                        f"volume {t.volume_month:,} > crossover {cx:,.0f} vs the '{tier}' API tier -> "
                        f"self-host undercuts (still within one box)")
    return Decision(ak, ap, "api", tier, contract,
                    f"volume {t.volume_month:,} <= crossover {cx:,.0f} vs the '{tier}' tier -> API is "
                    f"cheaper here; self-host would be a control/privacy choice, not a cost one")

TASKS = [
    Task("Support email classification",   "internal", "interactive", 2_000_000, "low",      True,  True,  False),
    Task("Clinical note summarisation",    "regulated","batch",         300_000, "frontier", False, False, False),
    Task("Internal policy Q&A",            "internal", "interactive",   500_000, "medium",   False, False, True),
    Task("Product-review sentiment@scale", "public",   "batch",       5_000_000, "medium",   True,  False, False),
]
import pandas as pd
rows = [{"task":t.name, **{k:getattr(decide(t),k) for k in ("approach","locus","tier","contract")}} for t in TASKS]
print(pd.DataFrame(rows).to_string(index=False))


                          task                                                                                   approach         locus        tier                                         contract
  Support email classification prompt baseline first; fine-tune is a CANDIDATE only if the measured baseline ceilings out           api       cheap                                              DPA
   Clinical note summarisation                                prompted LLM (measure the baseline before anything heavier) managed-cloud    frontier BAA/DPA + zero-retention (or keep it in-tenancy)
           Internal policy Q&A           RAG (the gap is missing facts, not wrong behaviour -> retrieve, don't fine-tune)           api         mid                                              DPA
Product-review sentiment@scale                                prompted LLM (measure the baseline before anything heavier)     self-host open-weight                                             none


**Read it.** Classification is low-capability, so its API tier is the *cheap* one — and against the cheap tier the self-host crossover sits **above one box's capacity**, so at 2M/mo the honest answer is **API**, not self-host. (The common 'high volume → self-host' reflex is backwards for a cheap task: self-host there is a control/privacy decision, not a cost win.) Clinical notes are regulated **and** need a frontier model → a frontier model *inside your tenancy* (managed-cloud), or a public API under a signed BAA/DPA — **not** 'no API regardless'. Policy Q&A has a knowledge gap → RAG. Sentiment at 5M/mo needs the *mid* tier, whose crossover is ~1.4M → above it, **self-host** genuinely undercuts. None of this reads the task name.


## 2. Prompt engineering is architecture — measure the baseline before you fine-tune

The discipline that stops a fine-tuning project you don't need: implement the best prompt, then **measure** it — with a confidence interval, because a small eval set lies if you read it as a point. Here we measure format compliance **and** accuracy for the classification task against local `qwen2.5:7b`. They are different bars: a reply can be perfectly formatted and wrong.


In [2]:
import sys, pathlib, importlib, math
here = pathlib.Path.cwd()
for p in (here, here/"notebooks", here.parent):
    if (p/"arch_kit.py").exists(): sys.path.insert(0, str(p)); break
import arch_kit; importlib.reload(arch_kit)
from arch_kit import chat

LABELS = {"billing","technical","account","other"}
SYS = ('You are a support-ticket classifier. Reply with ONLY a JSON object '
       '{"label": "<one of: billing, technical, account, other>"} and nothing else.')
# gold = the PRIMARY intent; the last two are deliberately ambiguous, so accuracy can diverge from format.
EVAL = [
    ("I was charged twice this month",                 "billing"),
    ("The app crashes when I open settings",           "technical"),
    ("How do I change my email address?",              "account"),
    ("Do you ship to Iceland?",                        "other"),
    ("My invoice total looks wrong",                   "billing"),
    ("Login returns a 500 error",                      "technical"),
    ("I can't log in and I want a refund for the lost days", "billing"),   # technical+billing -> judgement call
    ("Cancel my subscription and delete my account",   "account"),         # account+billing -> judgement call
]
import json as _json, re
def parse_label(txt):
    m = re.search(r"\{.*\}", txt, re.S)
    if not m: return None
    try:
        lab = _json.loads(m.group(0)).get("label")
        return lab.strip().lower() if isinstance(lab, str) else None   # normalise case/space
    except Exception:
        return None

def wilson(k, n, z=1.96):          # 95% lower/upper bound for k successes in n
    if n == 0: return (0.0, 0.0)
    p, d = k/n, 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return (max(0.0, c-h), min(1.0, c+h))

format_ok = correct = 0
for text, gold in EVAL:
    lab = parse_label(chat(text, system=SYS, num_predict=120))
    good = lab in LABELS
    format_ok += good
    correct  += (lab == gold)
    print(f"  [{'fmt-ok' if good else 'FMT-BAD'}] pred={str(lab):10} gold={gold:10} :: {text[:42]}")
n = len(EVAL)
fl, fu = wilson(format_ok, n); al, au = wilson(correct, n)
print(f"\nformat-compliance {format_ok}/{n}   95% CI [{fl:.0%}, {fu:.0%}]")
print(f"accuracy          {correct}/{n}   95% CI [{al:.0%}, {au:.0%}]")
print("n is small: read the interval, not the point. Fine-tuning is justified only if the bar you care")
print("about (format OR accuracy) stays low AFTER prompt work (schema, few-shot) and re-measurement.")


  [fmt-ok] pred=billing    gold=billing    :: I was charged twice this month


  [fmt-ok] pred=technical  gold=technical  :: The app crashes when I open settings


  [fmt-ok] pred=account    gold=account    :: How do I change my email address?


  [fmt-ok] pred=other      gold=other      :: Do you ship to Iceland?


  [fmt-ok] pred=billing    gold=billing    :: My invoice total looks wrong


  [fmt-ok] pred=technical  gold=technical  :: Login returns a 500 error


  [fmt-ok] pred=billing    gold=billing    :: I can't log in and I want a refund for the


  [fmt-ok] pred=account    gold=account    :: Cancel my subscription and delete my accou

format-compliance 8/8   95% CI [68%, 100%]
accuracy          8/8   95% CI [68%, 100%]
n is small: read the interval, not the point. Fine-tuning is justified only if the bar you care
about (format OR accuracy) stays low AFTER prompt work (schema, few-shot) and re-measurement.


**Read it.** Whatever those numbers are, they are now *measured*, with an interval that says how little 6-8 items can prove. Format and accuracy are separate gates: if format is high but accuracy is low, a fine-tune for *format* fixes the wrong problem. Note the two ambiguous tickets — the 'gold' there is a judgement call, and that is the point: your eval set encodes decisions, so it is itself an artifact to review, not ground truth handed down.


## 3. Cost model — compare against the tier that actually passes (assumptions explicit)

One reconciled price table (the `API_TIERS` constants from section 1). The crossover is computed **per tier** — the honest comparison is self-host vs the *cheapest tier that passes your eval*, not a mid-tier straw man. For a cheap-tier task the crossover can sit above one box's capacity, i.e. **there is no economic crossover at all** — then self-host is a control decision, full stop.


In [3]:
for tier in ("cheap", "mid", "frontier"):
    cx = crossover(tier)
    note = "ABOVE one box's capacity -> no economic crossover; self-host is a control/privacy call" \
           if cx > SELFHOST_CAPACITY_MONTH else "self-host undercuts this tier above the crossover"
    print(f"{tier:9} tier  crossover ~{cx:>12,.0f} req/mo   ({note})")

print()
for label, reqs, tier in [("classification@launch", 2_000_000, "cheap"),
                          ("classification@10x",   8_000_000, "cheap"),
                          ("sentiment@scale",       5_000_000, "mid"),
                          ("sentiment@peak",       12_000_000, "mid")]:
    a, s = api_cost(reqs, tier), selfhost_cost(reqs)
    print(f"{label:22} {reqs:>11,} req/mo  API({tier}) ${a:>9,.0f}  self-host ${s:>8,.0f}  cheaper: {'self-host' if s<a else 'API'}")


cheap     tier  crossover ~   9,523,810 req/mo   (ABOVE one box's capacity -> no economic crossover; self-host is a control/privacy call)
mid       tier  crossover ~   1,428,571 req/mo   (self-host undercuts this tier above the crossover)
frontier  tier  crossover ~     380,952 req/mo   (self-host undercuts this tier above the crossover)

classification@launch    2,000,000 req/mo  API(cheap) $      840  self-host $   4,000  cheaper: API
classification@10x       8,000,000 req/mo  API(cheap) $    3,360  self-host $   4,000  cheaper: API
sentiment@scale          5,000,000 req/mo  API(mid) $   14,000  self-host $   4,000  cheaper: self-host
sentiment@peak          12,000,000 req/mo  API(mid) $   33,600  self-host $   8,000  cheaper: self-host


### Hybrid routing — with the router and the misroute it really costs

Routing routine traffic to the cheap tier saves money, but it is **not free**: every request pays the router, and every misrouted request gets re-done on the frontier tier — so you pay *twice* for it. A saving quoted without those two terms is fiction.


In [4]:
ROUTER_PER_1K = 0.0002        # the router/classifier call, charged on EVERY request
def all_frontier_cost(reqs):
    return reqs * TOKENS_PER_REQUEST/1000 * API_TIERS["frontier"]
def routed_cost(reqs, frac_cheap, misroute):
    t = TOKENS_PER_REQUEST/1000
    router = reqs * t * ROUTER_PER_1K
    cheap  = reqs*frac_cheap     * t * API_TIERS["cheap"]
    front  = reqs*(1-frac_cheap) * t * API_TIERS["frontier"]
    redo   = reqs*frac_cheap*misroute * t * API_TIERS["frontier"]   # misrouted -> re-done on frontier
    return router + cheap + front + redo

reqs = 3_000_000
base = all_frontier_cost(reqs)
print(f"all-frontier baseline: ${base:,.0f}/mo\n")
for mis in (0.00, 0.05, 0.20):
    r = routed_cost(reqs, 0.70, mis)
    print(f"70% to cheap, misroute {mis:4.0%}  ->  ${r:>9,.0f}/mo   saving {100*(1-r/base):4.0f}%")
print("\nThe saving is hostage to two measured facts: the real routable fraction, and the misroute rate.")
print("A high misroute rate eats the saving because you pay the frontier price anyway, plus the cheap call.")


all-frontier baseline: $31,500/mo

70% to cheap, misroute   0%  ->  $   10,752/mo   saving   66%
70% to cheap, misroute   5%  ->  $   11,854/mo   saving   62%
70% to cheap, misroute  20%  ->  $   15,162/mo   saving   52%

The saving is hostage to two measured facts: the real routable fraction, and the misroute rate.
A high misroute rate eats the saving because you pay the frontier price anyway, plus the cheap call.


## 4. Meaningful tests

Each asserts a real decision rule on a **structured `Decision` field** (not a substring of prose), with its reason, and can fail — including property tests that must hold for *any* task and a generalisation test whose expected answer the cascade genuinely computes.


In [5]:
def run_checks():
    checks=[]
    def check(n,c,w): checks.append((n,"PASS" if c else "FAIL",w))
    cls, clinical, qa, senti = TASKS

    # --- approach rules ---
    check("knowledge gap -> RAG, not fine-tune",
          decide(qa).approach_kind == "rag",
          "a knowledge gap is missing facts -> retrieval; fine-tuning is the wrong tool for it")
    check("strict format + labels -> fine-tune is a CANDIDATE only",
          decide(cls).approach_kind == "finetune-candidate",
          "never credit fine-tuning before a measured prompted baseline")

    # --- locus / privacy, on structured fields ---
    check("regulated+frontier -> frontier IN-TENANCY, not banned",
          decide(clinical).locus == "managed-cloud" and "BAA/DPA" in decide(clinical).contract,
          "regulated data can travel under a BAA/DPA or stay in-tenancy; 'no API regardless' is false")
    check("low-capability task -> API tier is 'cheap', locus 'api'",
          decide(cls).locus == "api" and decide(cls).tier == "cheap",
          "a low-cap task uses the cheap tier, whose crossover is above one box -> no self-host cost win")
    check("mid-tier task above its crossover -> self-host",
          decide(senti).locus == "self-host",
          "5M/mo on the mid tier is past the ~1.4M crossover, so self-host genuinely undercuts")

    # --- cost model: crossover separates the two regimes (mid tier) ---
    cx = crossover("mid")
    check("self-host cheaper strictly above the crossover",
          selfhost_cost(int(cx*1.2)) < api_cost(int(cx*1.2),"mid")
          and api_cost(int(cx*0.8),"mid") < selfhost_cost(int(cx*0.8)),
          "the crossover must actually separate the regimes, or the model is decoration")
    check("a cheap-tier task has NO crossover below one box",
          crossover("cheap") > SELFHOST_CAPACITY_MONTH,
          "this is why 'high volume -> self-host' is false for cheap tasks")

    # --- hybrid routing, with overhead (not a tautology) ---
    check("routing beats all-frontier even with router+misroute",
          routed_cost(3_000_000,0.70,0.05) < all_frontier_cost(3_000_000),
          "the cheap tier carrying routine load wins AFTER paying the router and the misroute redo")
    check("misrouting erodes the saving",
          routed_cost(3_000_000,0.70,0.20) > routed_cost(3_000_000,0.70,0.05),
          "a higher misroute rate re-does more requests on the frontier tier -> less saving")

    # --- PROPERTY tests: must hold for ANY task ---
    import itertools
    def every_task():
        for pr,la,cap,gap in itertools.product(["public","internal","regulated"],
                ["batch","interactive","realtime"], ["low","medium","frontier"], [False,True]):
            yield Task("probe",pr,la,1_000_000,cap,True,True,gap)
    check("property: regulated is NEVER an unconditional public API",
          all(not(decide(t).locus=="api" and decide(t).contract=="none")
              for t in every_task() if t.privacy=="regulated"),
          "a regulated task must always carry a contract or stay in-tenancy")
    check("property: a knowledge gap NEVER resolves to fine-tune",
          all(decide(t).approach_kind != "finetune-candidate" for t in every_task() if t.knowledge_gap),
          "no attribute combination may turn a knowledge gap into a fine-tuning job")

    # --- GENERALISATION on an unseen task, expected answer COMPUTED from the rules ---
    wake = Task("Edge wake-word detection","public","realtime",50_000_000,"low",True,True,False)
    check("unseen realtime+low task -> on-device (latency gate), not a self-host LLM",
          decide(wake).locus == "on-device",
          "realtime + low capability is the on-device case; the old 'self-host LLM' answer ignored latency")
    tag = Task("Catalog image tagging","public","batch",4_000_000,"medium",False,False,False)
    check("unseen mid-tier high-volume task -> self-host (computed crossover)",
          decide(tag).locus == "self-host",
          "4M/mo on the mid tier is above the ~1.4M crossover; derived from attributes, not a name")

    import pandas as pd
    print(pd.DataFrame(checks, columns=["check","result","why it matters"]).to_string(index=False))
    assert all(r=="PASS" for _,r,_ in checks), "a model-selection rule broke"
    print("\nall", len(checks), "meaningful checks passed")
run_checks()


                                                                    check result                                                                                  why it matters
                                      knowledge gap -> RAG, not fine-tune   PASS             a knowledge gap is missing facts -> retrieval; fine-tuning is the wrong tool for it
                  strict format + labels -> fine-tune is a CANDIDATE only   PASS                                    never credit fine-tuning before a measured prompted baseline
                    regulated+frontier -> frontier IN-TENANCY, not banned   PASS      regulated data can travel under a BAA/DPA or stay in-tenancy; 'no API regardless' is false
                  low-capability task -> API tier is 'cheap', locus 'api'   PASS   a low-cap task uses the cheap tier, whose crossover is above one box -> no self-host cost win
                           mid-tier task above its crossover -> self-host   PASS             5M/mo on the mid tier 

## Check your understanding

**Q1.** Clinical-note summarisation needs a *frontier* model, yet the cascade did not just reach for a public API. What did the privacy gate actually decide — and why is 'regulated → no API ever' wrong?
<details><summary>answer</summary>The privacy gate decides *where the data may travel and under what contract*, not a blanket ban. Regulated + frontier resolves to a frontier model **inside your own cloud tenancy** (managed-cloud), or a public API **under a signed BAA/DPA with zero retention**. HIPAA permits PHI under a BAA, GDPR under a DPA; vendors sell exactly these. The gate constrains the *contract and locus*, and you take the best model you can run within that boundary.</details>

**Q2.** The support-classification task is high-volume, yet the cascade chose API, not self-host. Why does 'high volume → self-host' fail here?
<details><summary>answer</summary>Because it's a *low-capability* task, so it uses the **cheap** API tier. Against the cheap tier the self-host crossover sits *above* one box's capacity — there is no volume (within one box) where self-host undercuts. 'High volume → self-host' is only true when the task needs an expensive tier. For a cheap task, self-host is a control/privacy/latency decision, never a cost one. Always compare against the cheapest tier that passes your eval.</details>

**Q3.** The cost model says self-host beats API above a crossover volume. What makes that number soft — and when does the crossover not exist at all?
<details><summary>answer</summary>It's a function of the stated assumptions — the API tier's price, the box's monthly cost and capacity, tokens per request. Change any and it moves (that's why they're named constants). And for a cheap-tier task the crossover computes to *more requests than one box can serve*, so there is **no** crossover: self-host is never the cheaper option in that regime. A cost model is a way to reason about the trade, not a quote — never extrapolate linearly from pilot numbers.</details>

**Q4.** Hybrid routing showed a saving. What two measured facts is that saving hostage to, and what cost did a naive 'route 70% to cheap' estimate forget?
<details><summary>answer</summary>The real *routable fraction* and the *misroute rate*. The naive estimate forgets two line items: the **router call itself**, charged on every request, and the **misroute redo** — a request sent to the cheap tier that fails gets re-done on the frontier tier, so you pay twice. A high misroute rate eats the saving. The split and the misroute rate must be *measured* on real traffic with a quality check, not assumed.</details>

**Q5.** The test suite feeds in a wake-word task that isn't in the examples and asserts **on-device**. The earlier version asserted 'self-host LLM, fine-tune candidate' for the same task. Which is right, and what does that reveal about the test?
<details><summary>answer</summary>On-device is right. Wake-word detection is realtime with a tiny capability need — a network round-trip to any LLM API blows the latency budget, and it isn't an LLM task at all. The old assertion was *wrong*: it passed only because the code ignored latency and had no on-device branch. A generalisation test is only worth anything if its expected answer is the *correct* one, computed from the rules — a test that locks in a wrong answer is worse than no test.</details>
